# 🧠 Educational Guide: Training the MRI Dementia Classification Model

Welcome to the core ML component of the Early Dementia Detection system. This notebook is designed as a learning tool to guide you through the process of building a model that can classify brain MRI scans into four levels of impairment:
1. **No Impairment**
2. **Very Mild Impairment**
3. **Mild Impairment**
4. **Moderate Impairment**

### What we are doing here:
We are teaching a computer to look at 3D brain images and identify patterns of **atrophy** (shrinking of brain tissue) and **ventricular enlargement** (expansion of fluid-filled spaces), which are hallmark signs of dementia.

In [ ]:
import os
import torch
import numpy as np
import matplotlib.pyplot as plt
import monai
from monai.transforms import (Compose, LoadImage, EnsureChannelFirst, ScaleIntensity, 
                            NormalizeIntensity, Resize, EnsureType)
from monai.networks.nets import ResNet
from sklearn.metrics import confusion_matrix, classification_report

print("MONAI and PyTorch loaded successfully!")

## 📁 Step 1: Data Loading

Our images are stored in `backend/Datasets/` organized by folder name. We need to map these folder names to numerical labels (0-3) so the model can understand them.

In [ ]:
DATASET_DIR = "backend/Datasets/"
CLASSES = ["No Impairment", "Very Mild Impairment", "Mild Impairment", "Moderate Impairment"]

def get_image_paths():
    paths = []
    labels = []
    for idx, class_name in enumerate(CLASSES):
        class_path = os.path.join(DATASET_DIR, class_name)
        if os.path.exists(class_path):
            for img_file in os.listdir(class_path):
                paths.append(os.path.join(class_path, img_file))
                labels.append(idx)
    return paths, labels

paths, labels = get_image_paths()
print(f"Found {len(paths)} images across {len(CLASSES)} classes.")

## ⚙️ Step 2: The MRI Preprocessing Pipeline

Medical images are messy. They have noise, different sizes, and intensity variations. We apply a 4-step pipeline to ensure the model sees a 'standardized' brain.

1. **N4 Bias Field Correction**: MRI scans often have "shading" artifacts where one part of the image is brighter than another. N4 corrects this so the intensity is uniform.
2. **Skull Stripping**: We only care about the brain. We remove the skull, skin, and eyes to prevent the model from learning irrelevant patterns.
3. **MNI Registration**: Every human brain is shaped differently. We 'warp' all brains to a standard template (MNI152) so that the hippocampus is always in the same relative location for every patient.
4. **Z-Score Normalization**: We scale the pixel values so they have a mean of 0 and a variance of 1. This prevents the model's math from "exploding" during training.

In [ ]:
# Educational implementation of the pipeline using MONAI
preprocessing_transforms = Compose([
    LoadImage(image_only=True),
    EnsureChannelFirst(),
    # Note: Real N4 and Skull Stripping usually happen via external tools like ANTs or FSL
    # Here we use ScaleIntensity and NormalizeIntensity as proxies for the normalization step
    ScaleIntensity(),
    NormalizeIntensity(),
    Resize((128, 128, 128)), # Standardizing 3D volume size
    EnsureType()
])

print("Preprocessing pipeline defined.")

## 🏗️ Step 3: Model Architecture (3D-ResNet)

We use a **3D-ResNet**. Unlike a regular 2D image (like a photo), an MRI is a volume. A 3D Convolutional Neural Network (CNN) can "see" shapes across slices, allowing it to detect the volumetric loss of the hippocampus.

In [ ]:
model = ResNet(
    block=monai.networks.nets.ResNet.BasicBlock, 
    layers=[2, 3, 4, 6], 
    num_classes=4
)

print("3D-ResNet Model Initialized.")

## 🚀 Step 4: Training Loop and Evaluation

We use **Cross-Entropy Loss** because this is a classification task. We want the model to maximize the probability of the correct class.

In [ ]:
criterion = torch.nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)

def train_step(images, labels):
    model.train()
    optimizer.zero_grad()
    outputs = model(images)
    loss = criterion(outputs, labels)
    loss.backward()
    optimizer.step()
    return loss.item()

print("Training loop ready.")

## 📊 Step 5: Visualization and Results

To understand where the model is failing, we use a **Confusion Matrix**. For example, if the model often confuses "Very Mild" with "Mild", we know we need better data for those specific boundary cases.

In [ ]:
def plot_confusion_matrix(y_true, y_pred):
    cm = confusion_matrix(y_true, y_pred)
    plt.imshow(cm, interpolation='nearest', cmap=plt.cm.Blues)
    plt.xlabel('Predicted')
    plt.ylabel('True')
    plt.title('Dementia Impairment Confusion Matrix')
    plt.colorbar()
    plt.show()

print("Visualization functions ready.")